# 04 — Clasificación: Tipo de Demora (8 clases)

**Responsable:** Alberto Estrada Guerrero  
**Objetivo:** Encontrar el mejor clasificador multiclase con **Macro F1 ≥ 0.55**  
**Target:** `delay_label` — 8 clases de demora/cancelación

---

## Tabla de contenidos

1. [Setup y carga de datos](#1-setup)
2. [Exploración del target](#2-exploracion)
3. [Preparación de features](#3-features)
4. [Baselines](#4-baselines)
5. [Modelos principales](#5-modelos)
6. [Estrategias de balanceo](#6-balanceo)
7. [Optimización de hiperparámetros](#7-optimizacion)
8. [Evaluación del mejor modelo](#8-evaluacion)
9. [Análisis de errores](#9-errores)
10. [Exportar modelo y probabilidades para Dylan](#10-exportar)
11. [Registro de experimentos](#11-registro)

---
## 1. Setup y carga de datos <a id="1-setup"></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
import sys
import os
from pathlib import Path

# Agregar raíz del proyecto al path
sys.path.append(os.path.abspath(".."))

from config import (
    CLASSIFICATION_LABELS,
    SPLITS_DIR,
    RANDOM_STATE,
    TARGET_MACRO_F1,
    RESULTS_DIR,
)
from src.evaluation import (
    classification_metrics,
    plot_confusion_matrix,
    plot_per_class_f1,
)
from src.utils import set_seed, timer, df_summary

# Reproducibilidad
set_seed()
warnings.filterwarnings("ignore")

# Estilo de gráficos
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12
sns.set_style("whitegrid")

print(f"RANDOM_STATE: {RANDOM_STATE}")
print(f"TARGET_MACRO_F1: {TARGET_MACRO_F1}")
print(f"Clases: {CLASSIFICATION_LABELS}")

In [ ]:
# Cargar splits generados por Juan Camilo
train = pd.read_parquet(SPLITS_DIR / "train.parquet")
val = pd.read_parquet(SPLITS_DIR / "val.parquet")

print(f"Train: {train.shape[0]:,} filas x {train.shape[1]} columnas")
print(f"Val:   {val.shape[0]:,} filas x {val.shape[1]} columnas")
print(f"\nPeriodo train: {train['FlightDate'].min()} → {train['FlightDate'].max()}")
print(f"Periodo val:   {val['FlightDate'].min()} → {val['FlightDate'].max()}")

---
## 2. Exploración del target <a id="2-exploracion"></a>

Verificar distribución de las 8 clases y el desbalanceo esperado (~76% OnTime).

In [ ]:
# Distribución de clases en train
class_dist = train["delay_label"].value_counts()
class_pct = train["delay_label"].value_counts(normalize=True) * 100

dist_df = pd.DataFrame({"Conteo": class_dist, "Porcentaje": class_pct.round(2)})
dist_df = dist_df.reindex(CLASSIFICATION_LABELS)
print(dist_df)
print(f"\nTotal: {class_dist.sum():,}")

In [ ]:
# Gráfico de distribución de clases
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Barras absolutas
colors = sns.color_palette("Set2", n_colors=len(CLASSIFICATION_LABELS))
class_dist.reindex(CLASSIFICATION_LABELS).plot(
    kind="bar", ax=axes[0], color=colors, edgecolor="white"
)
axes[0].set_title("Distribución de clases (conteo)")
axes[0].set_ylabel("Número de vuelos")
axes[0].tick_params(axis="x", rotation=45)

# Barras porcentuales
class_pct.reindex(CLASSIFICATION_LABELS).plot(
    kind="bar", ax=axes[1], color=colors, edgecolor="white"
)
axes[1].set_title("Distribución de clases (%)")
axes[1].set_ylabel("Porcentaje")
axes[1].tick_params(axis="x", rotation=45)

for ax in axes:
    for container in ax.containers:
        ax.bar_label(container, fmt="%.0f", fontsize=9)

plt.tight_layout()
plt.savefig("../results/figures/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nRatio de desbalanceo (mayor/menor): {class_dist.max() / class_dist.min():.0f}x")

In [ ]:
# Distribución en validación (verificar que sea similar)
val_dist = val["delay_label"].value_counts(normalize=True) * 100

compare_dist = pd.DataFrame({
    "Train %": class_pct.reindex(CLASSIFICATION_LABELS),
    "Val %": val_dist.reindex(CLASSIFICATION_LABELS),
})
compare_dist["Diferencia pp"] = (compare_dist["Val %"] - compare_dist["Train %"]).round(2)
print("Comparación de distribución train vs val:")
print(compare_dist)

---
## 3. Preparación de features <a id="3-features"></a>

Separar features (X) del target (y). Identificar columnas no-feature.

In [ ]:
# Columnas que NO son features
NON_FEATURE_COLS = [
    "delay_label",       # target
    "FlightDate",        # fecha (ya codificada en features cíclicas)
    "dep_datetime_utc",  # datetime del join (no es feature)
]

# También excluir columnas de texto/identificadores si existen
ID_COLS = ["Reporting_Airline", "Origin", "Dest", "Marketing_Airline_Network",
           "OriginCityName", "DestCityName", "OriginState", "DestState",
           "airport", "airport_iata", "station"]

# Columnas a eliminar
drop_cols = [c for c in NON_FEATURE_COLS + ID_COLS if c in train.columns]

print(f"Columnas eliminadas: {drop_cols}")
print(f"Columnas restantes: {train.shape[1] - len(drop_cols)}")

In [ ]:
# Separar X e y
X_train = train.drop(columns=drop_cols, errors="ignore").select_dtypes(include=[np.number])
y_train = train["delay_label"]

X_val = val.drop(columns=drop_cols, errors="ignore").select_dtypes(include=[np.number])
y_val = val["delay_label"]

# Verificar NaNs
nan_pct = (X_train.isnull().sum() / len(X_train) * 100).sort_values(ascending=False)
cols_with_nans = nan_pct[nan_pct > 0]

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"\nFeatures ({X_train.shape[1]}):")
print(list(X_train.columns))

if len(cols_with_nans) > 0:
    print(f"\n⚠️ Columnas con NaN:")
    print(cols_with_nans.head(10))

In [ ]:
# Imputar NaN si los hay (mediana = robusto a outliers)
from sklearn.impute import SimpleImputer

if X_train.isnull().any().any():
    imputer = SimpleImputer(strategy="median")
    X_train = pd.DataFrame(
        imputer.fit_transform(X_train), 
        columns=X_train.columns, index=X_train.index
    )
    X_val = pd.DataFrame(
        imputer.transform(X_val),
        columns=X_val.columns, index=X_val.index
    )
    print(f"NaN imputados con mediana.")
    print(f"NaN restantes train: {X_train.isnull().sum().sum()}")
    print(f"NaN restantes val: {X_val.isnull().sum().sum()}")
    
    # Guardar imputer para reproducibilidad
    joblib.dump(imputer, "../models/classification/imputer.joblib")
else:
    imputer = None
    print("Sin NaN — no se necesita imputación.")

In [ ]:
# Correlación de features con el target (encoding ordinal temporal)
# Solo para exploración — no afecta el modelo
from sklearn.preprocessing import LabelEncoder

le_explore = LabelEncoder()
y_encoded_explore = le_explore.fit_transform(y_train)

correlations = X_train.corrwith(pd.Series(y_encoded_explore, index=X_train.index))
top_corr = correlations.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
top_corr.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 15 Features más correlacionadas con delay_label")
ax.set_xlabel("|Correlación|")
plt.tight_layout()
plt.show()

---
## 4. Baselines <a id="4-baselines"></a>

Dos baselines para tener un piso de referencia:
1. **DummyClassifier** (siempre predice la clase mayoritaria)
2. **Logistic Regression** con `class_weight='balanced'`

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline

# Encoder para modelos que necesitan labels numéricas
le = LabelEncoder()
le.fit(CLASSIFICATION_LABELS)
y_train_enc = le.transform(y_train)
y_val_enc = le.transform(y_val)

print(f"Mapping de clases:")
for i, label in enumerate(le.classes_):
    print(f"  {i} → {label}")

In [ ]:
# ─── Baseline 1: DummyClassifier (mayoritario) ───
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_val)

metrics_dummy = classification_metrics(y_val, y_pred_dummy)
print("=" * 50)
print("BASELINE: DummyClassifier (most_frequent)")
print("=" * 50)
print(f"  Accuracy:    {metrics_dummy['accuracy']:.4f}")
print(f"  Macro F1:    {metrics_dummy['macro_f1']:.4f}")
print(f"  Weighted F1: {metrics_dummy['weighted_f1']:.4f}")
print(f"  ¿Cumple target? {'✅' if metrics_dummy['meets_target'] else '❌'}")

In [ ]:
# ─── Baseline 2: Logistic Regression ───
pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
        multi_class="multinomial",
        solver="lbfgs",
    ))
])

pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_val)

metrics_lr = classification_metrics(y_val, y_pred_lr)
print("=" * 50)
print("BASELINE: Logistic Regression (balanced)")
print("=" * 50)
print(f"  Accuracy:    {metrics_lr['accuracy']:.4f}")
print(f"  Macro F1:    {metrics_lr['macro_f1']:.4f}")
print(f"  Weighted F1: {metrics_lr['weighted_f1']:.4f}")
print(f"  ¿Cumple target? {'✅' if metrics_lr['meets_target'] else '❌'}")

In [ ]:
# Confusion matrix del LogReg para ver patrón de errores
plot_confusion_matrix(y_val, y_pred_lr, title="Confusion Matrix — LogReg (baseline)")

---
## 5. Modelos principales <a id="5-modelos"></a>

Comparar Random Forest, XGBoost y LightGBM con sus estrategias de balanceo nativas.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# ─── Random Forest ───
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    max_depth=None,         # dejar que crezca completo
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_val)

metrics_rf = classification_metrics(y_val, y_pred_rf)
print("=" * 50)
print("Random Forest (class_weight=balanced)")
print("=" * 50)
print(f"  Accuracy:    {metrics_rf['accuracy']:.4f}")
print(f"  Macro F1:    {metrics_rf['macro_f1']:.4f}")
print(f"  Weighted F1: {metrics_rf['weighted_f1']:.4f}")
print(f"  ¿Cumple target? {'✅' if metrics_rf['meets_target'] else '❌'}")

In [ ]:
from xgboost import XGBClassifier

# ─── XGBoost ───
# Para multiclase, XGBoost no tiene class_weight directo.
# Usamos sample_weight calculado manualmente.
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight("balanced", y_train_enc)

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=len(CLASSIFICATION_LABELS),
    eval_metric="mlogloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
)
xgb.fit(X_train, y_train_enc, sample_weight=sample_weights)
y_pred_xgb = le.inverse_transform(xgb.predict(X_val))

metrics_xgb = classification_metrics(y_val, y_pred_xgb)
print("=" * 50)
print("XGBoost (sample_weight=balanced)")
print("=" * 50)
print(f"  Accuracy:    {metrics_xgb['accuracy']:.4f}")
print(f"  Macro F1:    {metrics_xgb['macro_f1']:.4f}")
print(f"  Weighted F1: {metrics_xgb['weighted_f1']:.4f}")
print(f"  ¿Cumple target? {'✅' if metrics_xgb['meets_target'] else '❌'}")

In [ ]:
from lightgbm import LGBMClassifier

# ─── LightGBM ───
lgbm = LGBMClassifier(
    n_estimators=200,
    max_depth=-1,
    learning_rate=0.1,
    is_unbalance=True,      # manejo nativo de desbalanceo
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
lgbm.fit(X_train, y_train_enc)
y_pred_lgbm = le.inverse_transform(lgbm.predict(X_val))

metrics_lgbm = classification_metrics(y_val, y_pred_lgbm)
print("=" * 50)
print("LightGBM (is_unbalance=True)")
print("=" * 50)
print(f"  Accuracy:    {metrics_lgbm['accuracy']:.4f}")
print(f"  Macro F1:    {metrics_lgbm['macro_f1']:.4f}")
print(f"  Weighted F1: {metrics_lgbm['weighted_f1']:.4f}")
print(f"  ¿Cumple target? {'✅' if metrics_lgbm['meets_target'] else '❌'}")

In [ ]:
# ─── Tabla comparativa de todos los modelos ───
all_results = {
    "DummyClassifier": metrics_dummy,
    "LogisticRegression": metrics_lr,
    "RandomForest": metrics_rf,
    "XGBoost": metrics_xgb,
    "LightGBM": metrics_lgbm,
}

comparison_df = pd.DataFrame([
    {
        "Modelo": name,
        "Accuracy": m["accuracy"],
        "Macro F1": m["macro_f1"],
        "Weighted F1": m["weighted_f1"],
        "Target ≥0.55": "✅" if m["meets_target"] else "❌",
    }
    for name, m in all_results.items()
]).sort_values("Macro F1", ascending=False).reset_index(drop=True)

print("\n" + "=" * 70)
print("COMPARATIVA DE MODELOS — PRIMERA RONDA")
print("=" * 70)
comparison_df

---
## 6. Estrategias de balanceo <a id="6-balanceo"></a>

Probar SMOTE y undersampling sobre el mejor modelo de la ronda anterior.
Comparar contra class_weight/is_unbalance que ya usamos.

In [ ]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek
from collections import Counter

print("Distribución original del train:")
print(Counter(y_train_enc))

In [ ]:
# ─── SMOTE ───
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train_enc)

print(f"Después de SMOTE:")
print(f"  Antes:  {len(X_train):,} → Después: {len(X_train_smote):,}")
print(f"  Distribución: {Counter(y_train_smote)}")

lgbm_smote = LGBMClassifier(
    n_estimators=200, learning_rate=0.1,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgbm_smote.fit(X_train_smote, y_train_smote)
y_pred_smote = le.inverse_transform(lgbm_smote.predict(X_val))

metrics_smote = classification_metrics(y_val, y_pred_smote)
print(f"\nLGBM + SMOTE:")
print(f"  Macro F1: {metrics_smote['macro_f1']:.4f}")
print(f"  Accuracy: {metrics_smote['accuracy']:.4f}")

In [ ]:
# ─── Random Undersampling ───
rus = RandomUnderSampler(random_state=RANDOM_STATE)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train_enc)

print(f"Después de Undersampling:")
print(f"  Antes:  {len(X_train):,} → Después: {len(X_train_under):,}")
print(f"  Distribución: {Counter(y_train_under)}")

lgbm_under = LGBMClassifier(
    n_estimators=200, learning_rate=0.1,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgbm_under.fit(X_train_under, y_train_under)
y_pred_under = le.inverse_transform(lgbm_under.predict(X_val))

metrics_under = classification_metrics(y_val, y_pred_under)
print(f"\nLGBM + Undersampling:")
print(f"  Macro F1: {metrics_under['macro_f1']:.4f}")
print(f"  Accuracy: {metrics_under['accuracy']:.4f}")

In [ ]:
# ─── SMOTE + Tomek Links (combinado) ───
smt = SMOTETomek(random_state=RANDOM_STATE)
X_train_smt, y_train_smt = smt.fit_resample(X_train, y_train_enc)

print(f"Después de SMOTE+Tomek:")
print(f"  Antes:  {len(X_train):,} → Después: {len(X_train_smt):,}")

lgbm_smt = LGBMClassifier(
    n_estimators=200, learning_rate=0.1,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgbm_smt.fit(X_train_smt, y_train_smt)
y_pred_smt = le.inverse_transform(lgbm_smt.predict(X_val))

metrics_smt = classification_metrics(y_val, y_pred_smt)
print(f"\nLGBM + SMOTE+Tomek:")
print(f"  Macro F1: {metrics_smt['macro_f1']:.4f}")
print(f"  Accuracy: {metrics_smt['accuracy']:.4f}")

In [ ]:
# ─── Comparativa de estrategias de balanceo ───
balance_results = {
    "LightGBM (is_unbalance)": metrics_lgbm,
    "LightGBM + SMOTE": metrics_smote,
    "LightGBM + Undersampling": metrics_under,
    "LightGBM + SMOTE+Tomek": metrics_smt,
}

balance_df = pd.DataFrame([
    {"Estrategia": name, "Macro F1": m["macro_f1"], "Accuracy": m["accuracy"]}
    for name, m in balance_results.items()
]).sort_values("Macro F1", ascending=False).reset_index(drop=True)

print("\n" + "=" * 60)
print("COMPARATIVA DE ESTRATEGIAS DE BALANCEO")
print("=" * 60)
balance_df

---
## 7. Optimización de hiperparámetros <a id="7-optimizacion"></a>

RandomizedSearchCV con TimeSeriesSplit sobre el mejor modelo + mejor estrategia de balanceo.

> **Nota:** Si SMOTE fue la mejor estrategia, aplicarla DENTRO de cada fold
> para evitar data leakage del balanceo.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

# Espacio de búsqueda para LightGBM
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [3, 5, 7, 10, 15, -1],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "num_leaves": [15, 31, 50, 80, 100],
    "min_child_samples": [5, 10, 20, 50, 100],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1.0],
    "reg_lambda": [0, 0.01, 0.1, 1.0],
}

# TimeSeriesSplit respeta el orden temporal
tscv = TimeSeriesSplit(n_splits=4)

# Modelo base (ajustar is_unbalance según resultado de la sección 6)
base_model = LGBMClassifier(
    is_unbalance=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_distributions,
    n_iter=50,                      # 50 combinaciones aleatorias
    scoring="f1_macro",
    cv=tscv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)

print("Iniciando RandomizedSearchCV (50 iteraciones × 4 folds = 200 fits)...")
search.fit(X_train, y_train_enc)

print(f"\n{'=' * 50}")
print(f"Mejor Macro F1 (CV): {search.best_score_:.4f}")
print(f"Mejores hiperparámetros:")
for param, value in search.best_params_.items():
    print(f"  {param}: {value}")

In [ ]:
# Evaluar mejor modelo en validación
best_clf = search.best_estimator_
y_pred_best = le.inverse_transform(best_clf.predict(X_val))

metrics_best = classification_metrics(y_val, y_pred_best)
print("=" * 50)
print("MEJOR MODELO OPTIMIZADO")
print("=" * 50)
print(f"  Accuracy:    {metrics_best['accuracy']:.4f}")
print(f"  Macro F1:    {metrics_best['macro_f1']:.4f}")
print(f"  Weighted F1: {metrics_best['weighted_f1']:.4f}")
print(f"  ¿Cumple target? {'✅' if metrics_best['meets_target'] else '❌'}")

In [ ]:
# Top 10 combinaciones de la búsqueda
cv_results = pd.DataFrame(search.cv_results_)
top10 = cv_results.nsmallest(10, "rank_test_score")[
    ["rank_test_score", "mean_test_score", "std_test_score", 
     "mean_train_score", "params"]
].rename(columns={
    "rank_test_score": "Rank",
    "mean_test_score": "Mean F1 (val)",
    "std_test_score": "Std F1",
    "mean_train_score": "Mean F1 (train)",
})

# Verificar sobreajuste: diferencia grande entre train y val = sobreajuste
top10["Gap (train-val)"] = top10["Mean F1 (train)"] - top10["Mean F1 (val)"]
top10

---
## 8. Evaluación del mejor modelo <a id="8-evaluacion"></a>

Visualizaciones detalladas del rendimiento del modelo optimizado.

In [ ]:
# Confusion Matrix normalizada (recall por clase)
plot_confusion_matrix(
    y_val, y_pred_best,
    title="Confusion Matrix — Mejor modelo (normalizada)",
    save_path="../results/figures/confusion_matrix_normalized.png",
    normalize=True,
)

In [ ]:
# Confusion Matrix absoluta (conteos)
plot_confusion_matrix(
    y_val, y_pred_best,
    title="Confusion Matrix — Mejor modelo (conteos)",
    save_path="../results/figures/confusion_matrix_absolute.png",
    normalize=False,
)

In [ ]:
# F1-Score por clase
plot_per_class_f1(
    y_val, y_pred_best,
    title="F1-Score por Clase — Mejor modelo",
    save_path="../results/figures/f1_per_class.png",
)

In [ ]:
# Classification report completo
from sklearn.metrics import classification_report

print("Classification Report completo:")
print("=" * 70)
print(classification_report(
    y_val, y_pred_best, 
    labels=CLASSIFICATION_LABELS,
    zero_division=0,
    digits=4,
))

In [ ]:
# Curvas Precision-Recall por clase (One-vs-Rest)
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.preprocessing import label_binarize

y_val_bin = label_binarize(y_val_enc, classes=range(len(CLASSIFICATION_LABELS)))
y_proba = best_clf.predict_proba(X_val)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (label, ax) in enumerate(zip(le.classes_, axes)):
    if i < y_val_bin.shape[1]:
        precision, recall, _ = precision_recall_curve(y_val_bin[:, i], y_proba[:, i])
        ap = average_precision_score(y_val_bin[:, i], y_proba[:, i])
        ax.plot(recall, precision, linewidth=2)
        ax.set_title(f"{label}\nAP={ap:.3f}", fontsize=10)
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
        ax.set_xlim([0, 1.05])
        ax.set_ylim([0, 1.05])
        ax.grid(True, alpha=0.3)

plt.suptitle("Curvas Precision-Recall por clase (One-vs-Rest)", fontsize=14)
plt.tight_layout()
plt.savefig("../results/figures/pr_curves.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 9. Análisis de errores <a id="9-errores"></a>

Entender dónde y por qué falla el modelo.

In [ ]:
# Feature importance
feat_imp = pd.Series(best_clf.feature_importances_, index=X_train.columns)
top20 = feat_imp.nlargest(20)

fig, ax = plt.subplots(figsize=(10, 8))
top20.sort_values().plot(kind="barh", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Top 20 Features — Clasificación")
ax.set_xlabel("Importancia")
plt.tight_layout()
plt.savefig("../results/figures/feature_importance_clf.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Accuracy por aeropuerto de origen
val_analysis = val.copy()
val_analysis["pred"] = y_pred_best
val_analysis["correct"] = (val_analysis["pred"] == val_analysis["delay_label"]).astype(int)

if "Origin" in val_analysis.columns:
    acc_by_airport = (
        val_analysis.groupby("Origin")["correct"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "Accuracy", "count": "Vuelos"})
        .sort_values("Accuracy")
    )
    
    fig, ax = plt.subplots(figsize=(10, 8))
    bars = ax.barh(acc_by_airport.index, acc_by_airport["Accuracy"], 
                   color="steelblue", edgecolor="white")
    ax.set_title("Accuracy por Aeropuerto de Origen")
    ax.set_xlabel("Accuracy")
    ax.axvline(x=metrics_best["accuracy"], color="red", linestyle="--", 
               label=f"Media = {metrics_best['accuracy']:.3f}")
    ax.legend()
    plt.tight_layout()
    plt.savefig("../results/figures/accuracy_by_airport.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    print("\nAeropuertos con peor accuracy:")
    print(acc_by_airport.head(5))

In [ ]:
# Errores por hora del día
if "CRSDepTime" in val_analysis.columns:
    val_analysis["hour"] = (val_analysis["CRSDepTime"] // 100).clip(0, 23)
    
    acc_by_hour = val_analysis.groupby("hour")["correct"].mean()
    
    fig, ax = plt.subplots(figsize=(12, 5))
    acc_by_hour.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_title("Accuracy por Hora del Día")
    ax.set_xlabel("Hora de salida programada")
    ax.set_ylabel("Accuracy")
    ax.axhline(y=metrics_best["accuracy"], color="red", linestyle="--")
    plt.tight_layout()
    plt.savefig("../results/figures/accuracy_by_hour.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ¿Qué clases confunde más? (errores más frecuentes)
errors = val_analysis[~val_analysis["correct"].astype(bool)]

if len(errors) > 0:
    confusion_pairs = (
        errors.groupby(["delay_label", "pred"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(10)
    )
    confusion_pairs.columns = ["Real", "Predicho", "Errores"]
    
    print("Top 10 pares de confusión más frecuentes:")
    print(confusion_pairs.to_string(index=False))

In [ ]:
# Distribución de probabilidad máxima en predicciones correctas vs incorrectas
max_proba = y_proba.max(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(max_proba[val_analysis["correct"] == 1], bins=50, alpha=0.6, 
        label="Correctas", color="green", density=True)
ax.hist(max_proba[val_analysis["correct"] == 0], bins=50, alpha=0.6, 
        label="Incorrectas", color="red", density=True)
ax.set_xlabel("Probabilidad máxima del modelo")
ax.set_ylabel("Densidad")
ax.set_title("Confianza del modelo: predicciones correctas vs incorrectas")
ax.legend()
plt.tight_layout()
plt.savefig("../results/figures/confidence_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Probabilidad media en aciertos: {max_proba[val_analysis['correct'] == 1].mean():.3f}")
print(f"Probabilidad media en errores:  {max_proba[val_analysis['correct'] == 0].mean():.3f}")

---
## 10. Exportar modelo y probabilidades para Dylan <a id="10-exportar"></a>

Guardar:
1. Mejor modelo serializado (`.joblib`)
2. Label encoder
3. Splits enriquecidos con probabilidades (para que Dylan las use como features)

In [ ]:
# ─── Guardar modelo ───
Path("../models/classification").mkdir(parents=True, exist_ok=True)

joblib.dump(best_clf, "../models/classification/best_clf.joblib")
joblib.dump(le, "../models/classification/label_encoder.joblib")
if imputer is not None:
    joblib.dump(imputer, "../models/classification/imputer.joblib")

print("Modelo guardado:  models/classification/best_clf.joblib")
print("Encoder guardado: models/classification/label_encoder.joblib")

In [ ]:
# ─── Generar probabilidades para Dylan ───
# Las probabilidades del clasificador se inyectan como features al regresor
proba_train = best_clf.predict_proba(X_train)
proba_val = best_clf.predict_proba(X_val)

proba_cols = [f"prob_{label}" for label in le.classes_]
print(f"Columnas de probabilidad: {proba_cols}")

# Agregar al dataset original
train_with_proba = train.copy()
for i, col in enumerate(proba_cols):
    train_with_proba[col] = proba_train[:, i]

val_with_proba = val.copy()
for i, col in enumerate(proba_cols):
    val_with_proba[col] = proba_val[:, i]

# Guardar
Path("../data/processed/splits").mkdir(parents=True, exist_ok=True)
train_with_proba.to_parquet("../data/processed/splits/train_with_proba.parquet", index=False)
val_with_proba.to_parquet("../data/processed/splits/val_with_proba.parquet", index=False)

print(f"\nArchivos para Dylan:")
print(f"  data/processed/splits/train_with_proba.parquet ({len(train_with_proba):,} filas)")
print(f"  data/processed/splits/val_with_proba.parquet ({len(val_with_proba):,} filas)")
print(f"\n⚡ Dylan: carga estos archivos y usa las columnas prob_* como features adicionales.")

In [ ]:
# Verificar las probabilidades
print("Estadísticas de las probabilidades exportadas (val):")
print(val_with_proba[proba_cols].describe().round(4))

# Verificar que suman ~1
proba_sums = val_with_proba[proba_cols].sum(axis=1)
print(f"\nSuma de probabilidades (debe ser ~1.0):")
print(f"  Media: {proba_sums.mean():.6f}")
print(f"  Min:   {proba_sums.min():.6f}")
print(f"  Max:   {proba_sums.max():.6f}")

---
## 11. Registro de experimentos <a id="11-registro"></a>

Resumen de todos los experimentos realizados. Copiar al `results/experiment_log.md`.

In [ ]:
# Resumen final de TODOS los experimentos
print("=" * 80)
print("RESUMEN FINAL — CLASIFICACIÓN")
print("=" * 80)

all_experiments = {
    "C01 DummyClassifier": metrics_dummy,
    "C02 LogisticRegression": metrics_lr,
    "C03 RandomForest": metrics_rf,
    "C04 XGBoost": metrics_xgb,
    "C05 LightGBM (is_unbalance)": metrics_lgbm,
    "C06 LightGBM + SMOTE": metrics_smote,
    "C07 LightGBM + Undersampling": metrics_under,
    "C08 LightGBM + SMOTE+Tomek": metrics_smt,
    "C09 LightGBM Optimizado": metrics_best,
}

final_df = pd.DataFrame([
    {
        "Exp": name,
        "Accuracy": f"{m['accuracy']:.4f}",
        "Macro F1": f"{m['macro_f1']:.4f}",
        "Weighted F1": f"{m['weighted_f1']:.4f}",
        "Target": "✅" if m["meets_target"] else "❌",
    }
    for name, m in all_experiments.items()
])

print(final_df.to_string(index=False))

# Guardar tabla
Path("../results/tables").mkdir(parents=True, exist_ok=True)
final_df.to_csv("../results/tables/classification_comparison.csv", index=False)
print(f"\nTabla guardada en results/tables/classification_comparison.csv")

In [ ]:
# Mejor configuración para la memoria
print("\n" + "=" * 60)
print("CONFIGURACIÓN DEL MEJOR MODELO (para la memoria)")
print("=" * 60)
print(f"\nModelo: LightGBM")
print(f"Hiperparámetros: {search.best_params_}")
print(f"Estrategia de balanceo: is_unbalance=True")
print(f"\nMétricas en validación (sep-oct 2024):")
print(f"  Accuracy:    {metrics_best['accuracy']:.4f}")
print(f"  Macro F1:    {metrics_best['macro_f1']:.4f}")
print(f"  Weighted F1: {metrics_best['weighted_f1']:.4f}")
print(f"\nRandom State: {RANDOM_STATE}")
print(f"Features usadas: {X_train.shape[1]}")
print(f"\n⚠️ RECORDATORIO: El test set (nov-dic 2024) NO se ha tocado.")
print(f"   Solo Héctor lo usará en el notebook 06_hierarchical_pipeline.ipynb")